In [ ]:
from finbourne_sdk_utils.jupyter_tools import toggle_code

"""Output Transactions

This notebook shows how LUSID uses synthetic transactions to fill in the gaps between user-instructed transactions and corporate actions.

Attributes
----------
instruments
transactions
portfolios
adjust holdings
output transactions
corporate actions
stock split
build transactions
sub-holding keys
"""

toggle_code("Hide docstring")

# Output Transactions Notebook


When LUSID generates a portfolio, set of holdings or a valuation it bases all calculations on the entire timeline of transactions within a portfolio. 

This is straightforward when a portfolio contains just user-added transactions, but there are many other ways that portfolios can be updated within LUSID.

This notebook shows how LUSID creates synthetic transactions within portfolios to represent non-transaction events such as Corporate Actions (e.g. bonus issue, dividend, stock split) or Holdings Adjustments. 

These synthetic transactions can be seen in the Output Transactions at the end of the notebook.

We will also explore how LUSID handles corporate actions on instruments that are being held under multiple strategies.


## Part 1: Setting up LUSID

To initialise our LUSID environment run the cell below



In [ ]:
# Import general purpose packages
import json
import os
import pandas as pd
from datetime import datetime

# Import lusid specific packages
# These are the core lusid packages for interacting with the API via Python
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
from finbourne.sdk.exceptions import ApiException
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne_sdk_utils.cocoon.cocoon import load_from_data_frame
from finbourne_sdk_utils.cocoon.cocoon_printer import (
    format_instruments_response,
    format_portfolios_response,
    format_transactions_response,
)
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.cocoon.utilities import create_scope_id

# Set pandas options
pd.set_option('display.max_columns', None)

# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook")

# Create a scope and portfolio code
scope = create_scope_id()
portfolio_code = "EQUITY_UK"
strategy_shk = "strategy"

### Load source transactions

We have a csv of transactions representing initial holdings of 4 FTSE100 stocks that we can use to initialise our portfolio. 

In [ ]:
df = pd.read_csv("data/equity_transactions_for_output.csv")
df

### Load instruments

Using the csv of setup transactions we can create instruments in LUSID for each of the equities referenced. When uploading the instruments we need to map certain properties that LUSID expects to columns provided in the csv.

For example, we need to map the 'ticker' column to the 'Isin' identifier and then 'sedol' to 'Sedol'.

In [ ]:
instrument_mapping = {
    "identifier_mapping": {
        "ClientInternal": "instrument_id",
        "Isin": "ticker",
        "Sedol": "sedol",
    },
    "required": {
        "name": "name"
    },
}

In [ ]:
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=df,
    mapping_required=instrument_mapping["required"],
    mapping_optional={},
    file_type="instruments",
    identifier_mapping=instrument_mapping["identifier_mapping"],
)

# Check the success rate of the upload
succ, failed, errors = format_instruments_response(result)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}])

### Create LUSID properties

Create the properties for the sub-holding keys to hold strategy information in our portfolio.

In [ ]:
properties_api = api_factory.build(lu.PropertyDefinitionsApi)

try:
    properties_api.create_property_definition(
        create_property_definition_request=models.CreatePropertyDefinitionRequest(
            domain="Transaction",
            scope=scope,
            code=strategy_shk,
            display_name=strategy_shk,
            life_time="Perpetual",
            value_required=False,
            data_type_id=models.ResourceId(scope="system", code="string")
        )
    )
except ApiException as e:
    detail = json.loads(e.body)
    if detail["code"] != 124: # 'PropertyAlreadyExists'
        raise e

### Create portfolio

To create the portfolio we will use the transaction csv as we did when creating instruments, mapping LUSID's required properties to columns in the dataframe, also providing a name, base currency and (optionally) setting the date at which the portfolio should be effective from (created).

In [ ]:
portfolio_mapping = {
    "required": {
        "code": "portfolio_code",
        "display_name": "portfolio_name",
        "base_currency": "$GBP",
    },
    "optional": {"created": "$2020-01-01T00:00:00+00:00"},
}

In [ ]:
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=df,
    mapping_required=portfolio_mapping["required"],
    mapping_optional=portfolio_mapping["optional"],
    file_type="portfolios",
    sub_holding_keys=[strategy_shk],
)

# Check the success rate of the upload
succ, failed = format_portfolios_response(result)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}])

### Load in transactions

Now that we have a list of instruments and a portfolio setup in LUSID, we can complete the initialisation by uploading the transactions.

The approach is similar to the instruments and portfolio.

In [ ]:
transaction_mapping = {
    "identifier_mapping": {
        "ClientInternal": "instrument_id",
    },
    "required": {
        "code": "portfolio_code",
        "transaction_id": "txn_id",
        "type": "txn_type",
        "transaction_price.price": "txn_price",
        "transaction_price.type": "$Price",
        "total_consideration.amount": "txn_consideration",
        "units": "txn_units",
        "transaction_date": "txn_trade_date",
        "total_consideration.currency": "portfolio_base_currency",
        "settlement_date": "txn_settle_date",        
    },
    "optional": {},
    "properties": ["strategy"],
}

In [ ]:
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=df,
    mapping_required=transaction_mapping["required"],
    mapping_optional=transaction_mapping["optional"],
    sub_holding_keys=[strategy_shk],
    file_type="transactions",
    identifier_mapping=transaction_mapping["identifier_mapping"],
    property_columns=transaction_mapping["properties"],
    properties_scope=scope,
)

# Check the success rate of the upload
succ, failed = format_transactions_response(result)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(failed)}])

### Check holdings

Now that we have initialised the portfolio, we can query LUSID to check our holdings.

The first thing we need to do is set up and call the transaction portfolio api.

In [ ]:
txn_portfolio_api = api_factory.build(lu.TransactionPortfoliosApi)

response = txn_portfolio_api.get_holdings(
    scope=scope,
    code=portfolio_code,
    property_keys=["Instrument/default/Name"],
)

LUSID python tools helps us transform the response into a dataframe where we can see a summary of our portfolio holdings.

We have:
- 1 portfolio
- 5 distinct holdings
    - across 4 unique instruments
    - assigned to 2 strategies 
    
The dataframe shows just one record for our Aviva holdings despite uploading multiple transactions involving Aviva equity. LUSID aggregated these into one holding as the transactions were assigned the same strategy ('balanced').

This is contrasted by the way LUSID handles the Barclays transactions. Our holdings show two records for Barclays, one for the 'balanced' strategy and another for the 'banking' one. 

The strategy property we see here is just one example of how LUSID can handle custom tags to transactions or holdings which makes it easy to view different slices of the portfolio depending on the user and their use case.



In [ ]:
with open(r"config/get_holdings_mapping.json") as mappings_file:
    get_holdings_json_mapping = json.load(mappings_file)

holdings_df = lusid_response_to_data_frame(
    response,
    rename_properties=True,
    column_name_mapping=get_holdings_json_mapping,
)

holdings_df

## Part 2: Adding Corporate Actions

We now want to process a selection of corporate actions that relate to instruments that we hold in our portfolio. 

Instead of having to calculate the results of the corporate action outside LUSID, and apply them individually to each affected portfolio, LUSID allows use of a corporate actions source to automatically apply the corporate actions to holdings across all portfolios.

We want to record the corporate actions in one place and have the option to apply them to every portfolio that holds the respective instruments. 

### Create a corporate actions source

The corporate actions source is a container for holding corporate actions that we can use to apply to any portfolio. 

You may want to use a different corporate action source for each of your data vendors.

In [ ]:
ca_source_code = "ca_vendor1"

try:
    source_request = models.CreateCorporateActionSourceRequest(
        scope=scope,
        code=ca_source_code,
        display_name=ca_source_code,
        description="Corporate Actions source for Output Transactions notebook",
    )
    
    source_result = api_factory.build(
        lu.CorporateActionSourcesApi
    ).create_corporate_action_source(create_corporate_action_source_request=source_request)
    
except:
    pass

To attach the corporate action source to the portfolio you can use the TransactionPortfolioApi to upsert the details.

In [ ]:
# Assign corporate actions source to our portfolio

api_factory.build(lu.TransactionPortfoliosApi).upsert_portfolio_details(
    scope=scope,
    code=portfolio_code,
    effective_at="2020-01-01T00:00:00+00:00",
    create_portfolio_details=models.CreatePortfolioDetails(
      corporate_action_source_id=models.ResourceId(
          scope=scope,
          code=ca_source_code,
      ),  
    ),
)

print(
    f"Corporate actions source of {ca_source_code} assigned to portfolio {portfolio_code}"
)

### Load transitions from an external source

Corporate Action Sources provide a list of Corporate Actions.  Corporate Actions contain a set of Transitions, that determine the instruments that are taking part in a corporate action, and what the effect of the corporate action is on holdings in that instrument.

In this example we are going to add transitions from a csv file for the following corporate actions:

* A bonus issue of 13,200 units from Aviva (1 bonus share per 10 held)
* A cash dividend of £425,000 from Barclays (£1.70 GBP per share held)
* A 2 for 1 stock split from Tesco (each 1 share held is split into 2 shares held)

These transitions are posted with a payment date of 31 March 2020.

In [ ]:
corporate_action_df = pd.read_csv("data/corp-acts/corpact_transitions_split.csv")

# Format the datetimes into ISO strings
for col in ["announcement_date", "ex_date", "payment_date", "record_date"]:
    corporate_action_df[col] = corporate_action_df[col].apply(
        lambda x: datetime.strptime(x, "%d/%m/%Y").strftime(format="%Y-%m-%dT00:00:00Z")
    )

corporate_action_df

For more information on corporate actions and transitions please see https://support.finbourne.com/how-are-corporate-actions-represented.

### Create the LUSID requests from the source data

We want to take the transitions from the dataframe and break each up into their component input and output objects in preparation for uploading into LUSID.

In [ ]:
# This code produces a dictionary of transitions where:
# The dict key is the corporate action code and the dict value is a list
# of input and output CorporateActionTransitionComponentRequest objects

transitions = {}
CLIENT_INSTRUMENT_IDENTIFIER = "Instrument/default/ClientInternal"

for index, ca in corporate_action_df.iterrows():
    
    # Create transition components
    cat_in = models.CorporateActionTransitionComponentRequest(
        instrument_identifiers={
            CLIENT_INSTRUMENT_IDENTIFIER: ca["client_id"]
        },
        units_factor=ca["input_units_factor"],
        cost_factor=ca["input_cost_factor"],
    )
    
    # Determine if the output is Cash or Stock, and create the 
    # appropriate transition component
    if "CCY" in str(ca["output_instrument_internal"]):
        cat_out = models.CorporateActionTransitionComponentRequest(
            instrument_identifiers={
                "Instrument/default/Currency": ca["output_instrument_internal"][4:]
            },
            units_factor=ca["output_units_factor"],
            cost_factor=ca["output_cost_factor"],
        )
    else:
        cat_out = models.CorporateActionTransitionComponentRequest(
            instrument_identifiers={
                CLIENT_INSTRUMENT_IDENTIFIER: ca["output_instrument_internal"]
            },
            units_factor=ca["output_units_factor"],
            cost_factor=ca["output_cost_factor"],
        )
        
    key = ca["code"]
    transitions.setdefault(key, [])
    transitions[key].append(ca["action_description"])
    transitions[key].append(cat_in)
    transitions[key].append(cat_out)

Using the broken-down transitions we want to create an [UpsertCorporateActionRequest](https://www.lusid.com/docs/api/#operation/BatchUpsertCorporateActions) for each corporate action.

In [ ]:
# Iterate through the transitions, turning them into LUSID Corporate Action Requests

for key, values in transitions.items():
    transition_code = key
    transition_type = values[0]
    transition_in = values[1]
    transitions_out = values[2:]
    
    for x in transitions_out:
        if isinstance(x, str):
            transitions_out.remove(x)
            
    temp_transition = models.CorporateActionTransitionRequest(
        input_transition=transition_in, output_transitions=transitions_out
    )
    
    # Extract the data for the corporate action from the LUID corporate actions dataframe
    data = corporate_action_df[corporate_action_df["code"] == transition_code]
    
    def upsert_car():
        upsert_car = models.UpsertCorporateActionRequest(
                    corporate_action_code=str(item["code"]),
                    announcement_date=item["announcement_date"],
                    ex_date=item["ex_date"],
                    record_date=item["record_date"],
                    payment_date=item["payment_date"],
                    transitions=[temp_transition])
        return upsert_car
    
    # Iterate through each row of data and create the Corporate Action
    # Request for the appropriate type of action.
    for row, item in data.iterrows():
        if transition_type == "dividend-cash":
            div_ca = upsert_car()
        if transition_type == "bonus-issue":
            bonus_ca = upsert_car()
        if transition_type == "stock-split":
            split_ca = upsert_car()
        
            

### Upsert the corporate actions into LUSID's movements engine

We now have 3 requests ready to upload to LUSID:
* div_ca
* bonus_ca
* split_ca

These are fed into the CorporateActionSourcesApi as a batch upsert of corporate actions.

In [ ]:
result = api_factory.build(
    lu.CorporateActionSourcesApi
).batch_upsert_corporate_actions(
    scope=scope, 
    code=ca_source_code, 
    upsert_corporate_action_request=[div_ca, bonus_ca, split_ca],
)

### View the updated holdings


The portfolio should now have:
* 145,200 units of Aviva as we received 1 bonus share for every 10 we held, resulting in an increase of 13,200 units
* £425,000 GBP cash as Barlcays paid out a dividend of £1.70 GBP for each of the 250,000 units in our portfolio
* 24,000 units of Tesco as each of our 12,000 units were split in two

In [ ]:
response = txn_portfolio_api.get_holdings(
    scope=scope,
    code=portfolio_code,
    property_keys=["Instrument/default/Name"],
)

holdings_df = lusid_response_to_data_frame(
    response,
    rename_properties=True,
    column_name_mapping=get_holdings_json_mapping,
)

holdings_df

You'll notice that the cash dividend is not being shown as one amount. As our Barclays equities are held across two strategies, the cash dividend is also assigned to the two strategies in the same ratio.

## Part 3: Manual holding adjustment

You can make manual adjustments to holdings in LUSID. For example, you may wish to do this if you know that there is a correction required between your source data and the data in LUSID e.g. transaction types not yet configured.

In time you would look to properly account for any mismatch.

### Adjust holdings to add an initial cash balance

Seeing the cash increase in the portfolio has highlighted the fact that there was no initial cash position on record. 

We can manually set the holdings of our portfolio to add in a starting cash balance of £500,000 GBP in our balanced strategy. We will make this effective from the same time the portfolio was created.

In [ ]:
# Set the date from which the cash balance will apply to be the start of the portfolio
response = txn_portfolio_api.get_details(
    scope=scope,
    code=portfolio_code,
)

holdings_effective_date = response.version.effective_from

# Define the actual initial cash balance
initial_cash_balance = 500000

# Create a holding adjustment to set our initial cash balance
holding_adjustment = [
    models.AdjustHoldingRequest( 
        instrument_identifiers={
            "Instrument/default/Currency": "GBP",
        },
        tax_lots=[
            models.TargetTaxLotRequest(
                units=initial_cash_balance,
            )
        ],
        sub_holding_keys={
            f"Transaction/{scope}/{strategy_shk}" : models.PerpetualProperty(
                key=f"Transaction/{scope}/{strategy_shk}",
                value=models.PropertyValue(label_value="balanced")),
            },
    )
] 

Once we have the adjustment set up, we can use the TransactionPortfoliosApi to update the portfolio in LUSID.

In [ ]:
set_holdings_response = api_factory.build(lu.TransactionPortfoliosApi).set_holdings(
    scope=scope,
    code=portfolio_code,
    effective_at=holdings_effective_date.isoformat(),
    adjust_holding_request=holding_adjustment,
)

### Check holdings post-adjustment

The portfolio should now have £925,000 GBP cash as we manually increased the cash position from £425,000 by £500,000

As we added the £500,000 to the 'balanced' strategy the cash balance for 'banking' has remained unchanged at £170,000.

In [ ]:
response = txn_portfolio_api.get_holdings(
    scope=scope,
    code=portfolio_code,
    property_keys=["Instrument/default/Name"],
)

holdings_df = lusid_response_to_data_frame(
    response,
    rename_properties=True,
    column_name_mapping=get_holdings_json_mapping,
)

holdings_df

## Part 4: What transactions make up our holdings?

Now we have the portfolio holdings looking correct thanks to a combination of initial transactions, corporate actions and a manual holding adjustment. 

LUSID calculates holdings using the Movements Engine and a set of input and output transactions, so what has it done to include the corporate actions and the adjustment?

Lets extract the list of transactions from LUSID to examine them a little closer.


In [ ]:
# Load a mapping file for DataFrame headers for the build transaction response
with open(r"config/build_transactions_mapping.json") as mappings_file:
    build_transactions_json_mapping = json.load(mappings_file)

# Query LUSID for the transactions that form our portfolio
build_transactions_response = txn_portfolio_api.build_transactions(
    scope=scope,
    code=portfolio_code,
    transaction_query_parameters=models.TransactionQueryParameters(
        start_date="2020-01-01", end_date="2020-12-31"
    ),
    property_keys=["Instrument/default/Name"],
)

build_transactions_df = lusid_response_to_data_frame(
    build_transactions_response,
    rename_properties=True,
    column_name_mapping=build_transactions_json_mapping,
)
build_transactions_df

We initialised the portfolio with 6 'StockIn' transactions which are visible as the first 6 items in the Output Transactions list. 

The remaining 5 transactions that we can see in the output are 'synthetic transactions' that LUSID creates in our portfolio as a result of the actions and adjustments that we have instructed. 

LUSID is built to ensure that anything that impacts a portfolio is represented, even if it has not been directly applied.

### Some things to note with transaction IDs

LUSID creates transaction IDs for synthetic transactions and rather than creating unique identifiers each time, it is designed to use IDs based on the source transaction or corporate action transition to retain the relationships and traceability. 

#### Bonus issue and stock split
We can see this in our notebook by looking at the transaction IDs for the bonus issue and stock split corporate actions (first dataframe). The IDs here match the 'code' or ID of the transitions that they are based on (second dataframe). 

In [ ]:
build_transactions_df.loc[build_transactions_df["TransactionType"] == "CorpActIncrease"]

In [ ]:
corporate_action_df.loc[corporate_action_df["code"] != 5943592342]

#### Holding adjustment

This is also true for the ID of the holding adjustment, however, in this case the ID used is simply the date-time that the adjustment was made. 

In [ ]:
build_transactions_df.loc[build_transactions_df["TransactionType"] == "AdjustmentIncrease"]

The general pattern is that there is a single 'input transaction' that has a unique identifier and all 'built 
transactions' relating to that input have the same identifier so that they can be linked. 

#### Dividend payment

This means that if we pro-rate a dividend payment across two sub-holding keys or if we split a transaction via strategy tags, each resultant transaction have the same id. We can see this by looking at the output transactions relating to our dividend payment from Barclays. The synthetic transactions that LUSID creates for each strategy have the same transaction IDs.

In [ ]:
build_transactions_df.loc[build_transactions_df["TransactionType"] == "Dividend"]

The transitions dataframe filtered on the dividend action:

In [ ]:
corporate_action_df.loc[corporate_action_df["code"] == 5943592342]